
 # Relatório Técnico & Analítico: LH Nautical


### **Programa:** Lighthouse Challenge — Indicium Academy

### **Objetivo:** Análise descritiva, tratamento de viés temporal, previsão de demanda de séries temporais e motor de recomendação colaborativo.
#

Este documento consolida a análise integrada de dados operacionais, financeiros e preditivos da LH Nautical. O estudo abrangeu a governança de dados nos canais de venda físico e digital, o desenvolvimento de um baseline de previsão de demanda para itens com alta sazonalidade e a implementação de um sistema de recomendação personalizado para estímulo de cross-selling.

O faturamento total auditado atingiu R$ 1,20 Bilhão, distribuído em 41.700 pedidos válidos (status paid ou confirmed). O canal digital (E-commerce) consolidou-se como o principal motor de volume financeiro, representando 70% da receita total (R$ 840,45 Milhões em 29.184 pedidos), enquanto as lojas físicas (POS) responderam por 30% do faturamento (R$ 358,92 Milhões em 12.516 pedidos).

### 📊 Painel Executivo (Power BI)

#### Performance de Canais e Médias do POS (Questão 5)
<img src="../imgs/pag1-bi.png" alt="Performance de Canais" width="850"/>

 1. Configuração do Ambiente e Carga dos Dados

Importação das bibliotecas essenciais e carregamento das bases transacionais e cadastrais.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Definição do diretório de dados
BASE_DIR = Path.cwd().parent
CSV_DIR = BASE_DIR / "1-lh_nautical_csv"

In [3]:
# Leitura dos datasets principais
products = pd.read_csv(CSV_DIR / "products.csv")
product_variants = pd.read_csv(CSV_DIR / "product_variants.csv")
orders = pd.read_csv(CSV_DIR / "orders.csv")
order_items = pd.read_csv(CSV_DIR / "order_items.csv")

In [4]:
print(f"Products: {products.shape}")
print(f"Variants: {product_variants.shape}")
print(f"Orders: {orders.shape}")
print(f"Order Items: {order_items.shape}")

Products: (500, 10)
Variants: (1009, 12)
Orders: (48998, 13)
Order Items: (147320, 8)


## 2. Módulo de Previsão de Demanda: Bússola de Bordo 702 (Questão 6)

### Contexto e Metodologia:

* **Objetivo:** Projetar as vendas do item no primeiro trimestre de 2026 (Q1/2026) utilizando um baseline de média móvel de 3 meses.

* **Prevenção de Data Leakage:** A série é agregada mensalmente com grid contínuo (preenchendo meses sem venda com 0). O cálculo da média móvel é defasado com `.shift(1)` para garantir que apenas meses passados componham a predição.

In [5]:
# 1. Identificar IDs do produto alvo
target_product = products[products["name"] == "Bússola de Bordo 702"]
product_id = target_product.iloc[0]["id"]
variant_ids = product_variants[product_variants["product_id"] == product_id][
    "id"
].tolist()

In [7]:
# 2. Filtrar pedidos válidos e unificar transações
valid_orders = orders[orders["status"].isin(["paid", "confirmed"])].copy()
valid_orders["placed_at"] = pd.to_datetime(valid_orders["placed_at"])

merged_sales = order_items[
    order_items["product_variant_id"].isin(variant_ids)
].merge(valid_orders, left_on="order_id", right_on="id")

In [8]:
# 3. Agregação mensal e criação da dimensão de datas contínua
merged_sales["year_month"] = merged_sales["placed_at"].dt.to_period("M")
monthly_sales = (
    merged_sales.groupby("year_month")["quantity"].sum().reset_index()
)

min_date = monthly_sales["year_month"].min()
max_date = pd.Period("2026-03", freq="M")
full_periods = pd.period_range(start=min_date, end=max_date, freq="M")

df_ts = (
    pd.DataFrame({"year_month": full_periods})
    .merge(monthly_sales, on="year_month", how="left")
    .fillna(0)
)
df_ts["quantity"] = df_ts["quantity"].astype(int)

In [9]:
# 4. Cálculo do Baseline (Média Móvel de 3 Meses sem vazamento)
df_ts["moving_avg_3m"] = df_ts["quantity"].shift(1).rolling(window=3).mean()

# 5. Avaliação no conjunto de Teste (Q1 2026)
test_periods = [
    pd.Period("2026-01", freq="M"),
    pd.Period("2026-02", freq="M"),
    pd.Period("2026-03", freq="M"),
]
test_df = df_ts[df_ts["year_month"].isin(test_periods)].copy()
test_df["absolute_error"] = np.abs(
    test_df["quantity"] - test_df["moving_avg_3m"]
)
mae = test_df["absolute_error"].mean()

test_df[
    ["year_month", "quantity", "moving_avg_3m", "absolute_error"]
].reset_index(drop=True)

# %%
print(f"Soma Total Prevista para Q1/2026: {test_df['moving_avg_3m'].sum():.2f}")
print(f"Soma Total Prevista (Arredondada): {round(test_df['moving_avg_3m'].sum())}")
print(f"Erro Médio Absoluto (MAE): {mae:.2f}")

Soma Total Prevista para Q1/2026: 95.33
Soma Total Prevista (Arredondada): 95
Erro Médio Absoluto (MAE): 18.67


### Diagnóstico Analítico (Séries Temporais)

* **Adequação:** O baseline simples **não é adequado** para o segmento náutico devido à alta **sazonalidade de verão** observada em janeiro (66 vendas reais vs. 21,33 previstas).

* **Limitação Técnica:** O método sofre de **defasagem temporal**, reagindo tardiamente a picos de demanda e gerando risco de ruptura de estoque no início da temporada e acúmulo de estoque no final.

## 3. Motor de Recomendação: Motor de Popa 1949 (Questão 7)

### Contexto e Metodologia:

* **Objetivo:** Criar um motor de recomendação baseado em filtragem colaborativa item-item (*Item-Based Collaborative Filtering*).

* **Estrutura:** Matriz binária de interação Usuário $\times$ Produto (1 = comprou ao menos uma vez, 0 = não comprou), seguida de Similaridade de Cosseno entre os vetores de cada produto.

In [10]:
# 1. Obter ID de referência do Motor de Popa 1949
target_name = "Motor de Popa 1949"
target_id = products[products["name"] == target_name].iloc[0]["id"]

In [12]:
# 2. Junção da esteira transacional
items_variants = order_items.merge(
    product_variants[["id", "product_id"]],
    left_on="product_variant_id",
    right_on="id",
)

df_rec = items_variants.merge(
    valid_orders[["id", "customer_id"]], left_on="order_id", right_on="id"
)

In [13]:
# 3. Construção da Matriz de Interação Usuário x Produto (Binarizada)
user_product_matrix = (
    pd.crosstab(df_rec["customer_id"], df_rec["product_id"]) > 0
).astype(int)

In [14]:
# 4. Cálculo da Similaridade de Cosseno entre Produtos
product_user_matrix = user_product_matrix.T
cosine_sim_matrix = cosine_similarity(product_user_matrix)

sim_df = pd.DataFrame(
    cosine_sim_matrix,
    index=product_user_matrix.index,
    columns=product_user_matrix.index,
)

In [15]:
# 5. Extração do Top 5 mais similares (excluindo o próprio item)
target_sims = (
    sim_df[target_id].drop(index=target_id).reset_index()
)
target_sims.columns = ["product_id", "similarity_score"]

top_5 = (
    target_sims.merge(
        products[["id", "name"]], left_on="product_id", right_on="id"
    )
    .sort_values(by="similarity_score", ascending=False)
    .head(5)
    .reset_index(drop=True)
)

top_5[["name", "similarity_score"]]

,name,similarity_score
0,Vela Mestra 1913,0.245200
1,Cabo Náutico 2105,0.229962
2,GPS Plotter 2249,0.214818
3,Motor de Popa 1540,0.212121
4,Vela Mestra 3870,0.208824


### 📊 Painel Executivo (Power BI)

#### Previsão de Demanda e Motor de Recomendação (Questões 4, 6 e 7)
<img src="../imgs/pag2-bi.png" alt="Performance de Canais" width="850"/>

### Diagnóstico Analítico (Recomendação)

* **Item Mais Similar:** **Vela Mestra 1913** (Score de Similaridade: 0,2452).

* **Significado do Score:** Mede a proximidade angular da base de clientes compradores. Um score maior indica que há alta sobreposição entre quem comprou o motor e quem comprou o produto recomendado.

* **Limitação do Modelo:** Novos produtos ou clientes sem histórico não possuem interações na matriz, resultando em similaridade zero e impedindo recomendações automáticas imediatas.

## 4. Conclusão & Ações Estratégicas

**Cross-Selling na Vitrine:** Integrar o Top 5 no checkout ao adicionar o *Motor de Popa 1949*, especialmente acessórios de amarração e navegação (*Cabo Náutico 2105* e *GPS Plotter 2249*).

**Evolução do Modelo Preditivo:** Substituir a média móvel por modelos sazonais com antecedência de compra de 60 dias para o período de alta temporada (novembro a janeiro).

**Gestão de Canais:** Manter o monitoramento via dashboard no Power BI utilizando dimensões contínuas de data para evitar distorções nas tomadas de decisão da diretoria.